# 02 — Load Raw Plate Data

Loads all raw plate CSV files, assigns well types from metadata, and saves a unified long-format dataset.

**Requires:** `data/plate_metadata.json` (run `01_plate_metadata.ipynb` first)

In [1]:
import json
import pandas as pd
import os

with open('data/plate_metadata.json') as f:
    meta = json.load(f)

print(f"Assay: {meta['assay_name']}")
print(f"Format: {meta['plate_format']}")
print(f"Plates to load: {meta['num_plates']}")

Assay: HTS Fluorescence Assay
Format: 384-well
Plates to load: 5


## Load and validate each plate

In [2]:
all_records = []

for i, fname in enumerate(meta['plate_files'], 1):
    fpath = os.path.join('data', 'raw', fname)
    plate = pd.read_csv(fpath, index_col='Row')
    plate.columns = plate.columns.astype(int)

    # Validate shape
    assert plate.shape == (16, 24), f'{fname}: unexpected shape {plate.shape}'

    # Melt to long format
    for row in plate.index:
        for col in plate.columns:
            well = f'{row}{col:02d}'
            all_records.append({
                'Plate': f'Plate {i}',
                'PlateFile': fname,
                'Row': row,
                'Column': col,
                'Well': well,
                'WellType': meta['well_types'][well],
                'RFU': int(plate.at[row, col])
            })

    print(f'Plate {i}: loaded {plate.shape[0]}x{plate.shape[1]} — OK')

df = pd.DataFrame(all_records)
print(f'\nTotal wells loaded: {len(df)}')
print(df.groupby(['Plate', 'WellType'])['Well'].count().unstack())

Plate 1: loaded 16x24 — OK
Plate 2: loaded 16x24 — OK
Plate 3: loaded 16x24 — OK
Plate 4: loaded 16x24 — OK
Plate 5: loaded 16x24 — OK

Total wells loaded: 1920
WellType  Experimental  High  Low  Medium
Plate                                    
Plate 1            320    24   24      16
Plate 2            320    24   24      16
Plate 3            320    24   24      16
Plate 4            320    24   24      16
Plate 5            320    24   24      16


## Preview data

In [3]:
df.head(10)

,Plate,PlateFile,Row,Column,Well,WellType,RFU
0,Plate 1,sample_384_plate.csv,A,1,A01,Low,339
1,Plate 1,sample_384_plate.csv,A,2,A02,Medium,9723
2,Plate 1,sample_384_plate.csv,A,3,A03,Experimental,20181
3,Plate 1,sample_384_plate.csv,A,4,A04,Experimental,27184
4,Plate 1,sample_384_plate.csv,A,5,A05,Experimental,13126
5,Plate 1,sample_384_plate.csv,A,6,A06,Experimental,13126
6,Plate 1,sample_384_plate.csv,A,7,A07,Experimental,27633
7,Plate 1,sample_384_plate.csv,A,8,A08,Experimental,21139
8,Plate 1,sample_384_plate.csv,A,9,A09,Experimental,11244
9,Plate 1,sample_384_plate.csv,A,10,A10,Experimental,19340


## Save unified dataset

In [4]:
out = 'data/processed/all_plates_raw.csv'
df.to_csv(out, index=False)
print(f'Saved: {out} ({len(df)} rows)')

Saved: data/processed/all_plates_raw.csv (1920 rows)
